# 📊 Exploratory Data Analysis - Loan Default Prediction

This notebook performs comprehensive exploratory data analysis on loan data to understand patterns, relationships, and insights that will inform our machine learning model development.

## 🎯 Objectives
1. Understand data structure and quality
2. Analyze feature distributions and relationships
3. Examine default rates across different segments
4. Identify key risk factors
5. Generate insights for feature engineering

## 📋 Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Import our custom modules
import sys
sys.path.append('../src')
from data_utils import generate_sample_data, validate_loan_data, get_data_summary
from feature_engineering import engineer_all_features

print("✅ All libraries imported successfully!")

## 📊 Data Loading and Overview

In [ ]:
# Generate sample data for analysis
df = generate_sample_data(n_samples=5000, random_state=42)
print(f"Generated dataset with {len(df)} samples and {len(df.columns)} features")
print(f"Shape: {df.shape}")
print()

# Display first few rows
print("First 5 rows:")
df.head()

In [ ]:
# Basic data information
print("Dataset Info:")
df.info()
print()

# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Data validation
validation_results = validate_loan_data(df)
print("Data Validation Results:")
print(f"Valid: {validation_results['is_valid']}")
if validation_results['warnings']:
    print("Warnings:")
    for warning in validation_results['warnings']:
        print(f"  - {warning}")
if validation_results['errors']:
    print("Errors:")
    for error in validation_results['errors']:
        print(f"  - {error}")

# Data summary
summary = get_data_summary(df)
print("\nData Summary:")
for key, value in summary.items():
    print(f"{key}: {value}")

## 📈 Target Variable Analysis

In [ ]:
# Default rate analysis
default_rate = df['default'].mean()
print(f"Overall default rate: {default_rate:.1%}")
print(f"Total defaults: {df['default'].sum():,}")
print(f"Total non-defaults: {(df['default'] == 0).sum():,}")

# Visualize class distribution
fig = px.pie(
    values=df['default'].value_counts(),
    names=['No Default', 'Default'],
    title=f'Loan Default Distribution (Default Rate: {default_rate:.1%})',
    color_discrete_map={'No Default': 'lightgreen', 'Default': 'lightcoral'}
)
fig.show()

## 🔍 Feature Distribution Analysis

In [ ]:
# Numerical features distribution
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numerical_cols.remove('default')  # Remove target variable

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=numerical_cols,
    vertical_spacing=0.1
)

for i, col in enumerate(numerical_cols[:9]):  # Show first 9
    row = i // 3 + 1
    col_idx = i % 3 + 1
    
    # Add histogram
    fig.add_trace(
        go.Histogram(x=df[col], name=col, nbinsx=30),
        row=row, col=col_idx
    )

fig.update_layout(height=900, title_text="Numerical Features Distribution")
fig.show()

In [ ]:
# Categorical features analysis
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

if categorical_cols:
    fig, axes = plt.subplots(1, len(categorical_cols), figsize=(15, 5))
    if len(categorical_cols) == 1:
        axes = [axes]
    
    for i, col in enumerate(categorical_cols):
        df[col].value_counts().plot(kind='bar', ax=axes[i])
        axes[i].set_title(f'{col} Distribution')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features found in the dataset")

## 📊 Default Rate Analysis by Features

In [ ]:
# Function to analyze default rates by feature
def analyze_default_rate_by_feature(df, feature, bins=None):
    """Analyze default rate by feature with optional binning"""
    if bins and pd.api.types.is_numeric_dtype(df[feature]):
        # Create bins for numerical features
        df_temp = df.copy()
        df_temp[f'{feature}_bin'] = pd.cut(df_temp[feature], bins=bins)
        analysis = df_temp.groupby(f'{feature}_bin')['default'].agg(['count', 'sum', 'mean']).reset_index()
        analysis.columns = [feature, 'total_count', 'default_count', 'default_rate']
    else:
        # Group by categorical features
        analysis = df.groupby(feature)['default'].agg(['count', 'sum', 'mean']).reset_index()
        analysis.columns = [feature, 'total_count', 'default_count', 'default_rate']
    
    return analysis

# Analyze default rates by key features
key_features = ['credit_score', 'income', 'loan_amount', 'debt_to_income', 'employment_length']

for feature in key_features:
    if feature in df.columns:
        analysis = analyze_default_rate_by_feature(df, feature, bins=5)
        print(f"\n{feature.upper()} vs Default Rate:")
        print(analysis)
        print()

In [ ]:
# Visualize default rates by credit score ranges
df['credit_score_bin'] = pd.cut(df['credit_score'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
credit_analysis = df.groupby('credit_score_bin')['default'].agg(['count', 'mean']).reset_index()

fig = px.bar(
    credit_analysis,
    x='credit_score_bin',
    y='mean',
    text='mean',
    title='Default Rate by Credit Score Range',
    labels={'mean': 'Default Rate', 'credit_score_bin': 'Credit Score Range'},
    hover_data=['count']
)
fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.show()

In [ ]:
# Income vs Default Rate
df['income_bin'] = pd.cut(df['income'], bins=5, labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
income_analysis = df.groupby('income_bin')['default'].agg(['count', 'mean']).reset_index()

fig = px.bar(
    income_analysis,
    x='income_bin',
    y='mean',
    text='mean',
    title='Default Rate by Income Range',
    labels={'mean': 'Default Rate', 'income_bin': 'Income Range'},
    hover_data=['count']
)
fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.show()

## 🔗 Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = df[numerical_cols + ['default']].corr()

# Create correlation heatmap
fig = px.imshow(
    correlation_matrix,
    text_auto=True,
    aspect="auto",
    title='Feature Correlation Matrix',
    color_continuous_scale='RdBu_r',
    zmid=0
)
fig.show()

# Show correlations with target
target_correlations = correlation_matrix['default'].drop('default').sort_values(key=abs, ascending=False)
print("Correlations with Default (Target):")
print(target_correlations)

## 🏗️ Feature Engineering Preview

In [ ]:
# Apply feature engineering
df_engineered = engineer_all_features(df)

print(f"Original features: {len(df.columns)}")
print(f"Engineered features: {len(df_engineered.columns)}")
print(f"New features added: {len(df_engineered.columns) - len(df.columns)}")

# Show new features
new_features = [col for col in df_engineered.columns if col not in df.columns]
print(f"\nNew features created:")
for feature in new_features[:10]:  # Show first 10
    print(f"  - {feature}")
if len(new_features) > 10:
    print(f"  ... and {len(new_features) - 10} more")

In [ ]:
# Analyze correlation of new features with target
new_feature_correlations = df_engineered[new_features + ['default']].corr()['default'].drop('default').sort_values(key=abs, ascending=False)

print("Top 10 New Features by Correlation with Default:")
print(new_feature_correlations.head(10))

# Visualize top correlations
fig = px.bar(
    x=new_feature_correlations.head(10).values,
    y=new_feature_correlations.head(10).index,
    orientation='h',
    title='Top 10 Engineered Features by Correlation with Default',
    labels={'x': 'Correlation', 'y': 'Feature'}
)
fig.show()

## 📋 Key Insights Summary

In [ ]:
# Generate insights summary
print("🔍 KEY INSIGHTS FROM EDA:")
print("=" * 50)
print()

print("📊 DATASET OVERVIEW:")
print(f"• Total samples: {len(df):,}")
print(f"• Default rate: {df['default'].mean():.1%}")
print(f"• Numerical features: {len(numerical_cols)}")
print(f"• Categorical features: {len(categorical_cols)}")
print()

print("🎯 TARGET VARIABLE:")
print(f"• Class imbalance ratio: {(df['default'] == 0).sum() / (df['default'] == 1).sum():.1f}:1")
print(f"• This indicates a moderate class imbalance that may require handling")
print()

print("🔑 KEY RISK FACTORS IDENTIFIED:")

# Credit score analysis
low_credit = df[df['credit_score'] < 600]['default'].mean()
high_credit = df[df['credit_score'] > 700]['default'].mean()
print(f"• Credit Score Impact: Low credit (<600) has {low_credit:.1%} default rate vs {high_credit:.1%} for high credit (>700)")

# Income analysis
low_income = df[df['income'] < 30000]['default'].mean()
high_income = df[df['income'] > 75000]['default'].mean()
print(f"• Income Impact: Low income (<30k) has {low_income:.1%} default rate vs {high_income:.1%} for high income (>75k)")

# DTI analysis
high_dti = df[df['debt_to_income'] > 40]['default'].mean()
low_dti = df[df['debt_to_income'] < 20]['default'].mean()
print(f"• Debt-to-Income Impact: High DTI (>40%) has {high_dti:.1%} default rate vs {low_dti:.1%} for low DTI (<20%)")
print()

print("🏗️ FEATURE ENGINEERING OPPORTUNITIES:")
print(f"• Created {len(new_features)} new features through engineering")
print(f"• Top engineered feature correlation: {new_feature_correlations.iloc[0]:.3f}")
print(f"• Financial ratios and risk scores show strong predictive power")
print()

print("⚠️ DATA QUALITY CONSIDERATIONS:")
print("• No missing values detected in generated data")
• All numerical features within reasonable ranges")
• Class balance requires attention for model training")
print()

print("🎯 MODEL DEVELOPMENT RECOMMENDATIONS:")
print("• Focus on recall optimization due to high cost of missing defaults")
• Consider ensemble methods to handle class imbalance")
• Feature engineering will be crucial for model performance")
• Implement proper cross-validation with stratified sampling")
• Use SHAP for model interpretability in production")
print()

print("✅ EDA Complete! Ready for model development.")

## 💾 Save Processed Data

Save the engineered dataset for model training:

In [ ]:
# Save processed data
import os
os.makedirs('../data/processed', exist_ok=True)
df_engineered.to_csv('../data/processed/loan_data_engineered.csv', index=False)
df.to_csv('../data/processed/loan_data_original.csv', index=False)

print("✅ Data saved successfully!")
print("Files saved:")
print("- ../data/processed/loan_data_engineered.csv")
print("- ../data/processed/loan_data_original.csv")